# Inicialização

In [1]:
from pathlib import Path
import os
import logging
import sys

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

## ---------LOGGING-----------
# Define paths based on environment
if Path("/kaggle").exists():
    logging.info("IN KAGGLE")
    os.environ["AMBIENTE"] = "KAGGLE"
    os.environ["TENSORBOARD_NO_TF"] = "1"

    PATH_DATASET = Path("/kaggle/working/TRAB_SERIES_TEMP")
    PATH_CODE = PATH_DATASET / "src"
    PATH_OUTPUT_DIR = PATH_DATASET / "outputs"
    EXPORT_DIR = PATH_OUTPUT_DIR / "export"
elif Path("/content").exists():
    os.environ["AMBIENTE"] = "COLAB"
    # PATH_DATASET = Path("/content/DELETAR")
    # PATH_CODE = PATH_DATASET / "src"
    # PATH_OUTPUT_DIR = PATH_DATASET / "outputs"
    # EXPORT_DIR = PATH_OUTPUT_DIR / "export"
    raise Exception
else:
    os.environ["AMBIENTE"] = "LOCAL"
    PATH_CODE = Path.cwd()
    PATH_DATASET = PATH_CODE
    PATH_OUTPUT_DIR = PATH_DATASET / "outputs"
    # Add to setup cell
    EXPORT_DIR = PATH_OUTPUT_DIR / "export"

assert type(PATH_DATASET) is not str, "PATH_DATASET NÃO DEVE SER STR!"
# Check if installation has been done
INSTALL_MARKER = PATH_DATASET / ".install_complete"

try:
    if not INSTALL_MARKER.exists():
        # Environment-specific setup
        if os.environ["AMBIENTE"] == "KAGGLE":
            import kaggle_secrets

            os.system("pip install uv")

            # user_secrets = kaggle_secrets.UserSecretsClient()
            # github_pat = user_secrets.get_secret("GITHUB_PAT")

            os.chdir("/kaggle/working")
            os.system(
                "git clone -b class-imbalance https://github.com/lfaoliveira/TRAB_SERIES_TEMP.git"
            )
            os.chdir(PATH_DATASET)

        # Install dependencies
        os.chdir(PATH_DATASET)
        os.system("uv pip install --requirements pyproject.toml --system")

        if os.environ["AMBIENTE"] == "KAGGLE":
            os.system(
                "uv pip install --upgrade --force-reinstall --no-cache-dir scipy numpy matplotlib protobuf tensorboard"
            )

        # Mark installation as complete
        INSTALL_MARKER.touch()
        logging.info("Installation completed")
    else:
        logging.info("Installation already completed, skipping...")

    os.chdir(PATH_CODE)
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    logging.info(f"Current working directory: {os.getcwd()}")
except Exception as e:
    logging.info("FALHA AO INICIAR NOTEBOOK")
    logging.info(e)

INFO:root:Installation already completed, skipping...
INFO:root:Current working directory: c:\Users\LUIS FELIPE\Desktop\SERIES_TEMP


## Carregamento + Baselines

In [ ]:
import gc

import pandas as pd

from src.data.dataset import NasaDataset

# from src.data.dataset import NasaDataset
# from src.data.datamodule import StrokeDataModule
import warnings

warnings.filterwarnings("ignore")

# Instanciar dataset (carrega e preprocessa automaticamente)
print("Carregando StrokeDataset...")
PATH_DATA = Path("data")
PATH_EXPORT = Path("outputs", "export")
dataset = NasaDataset(PATH_DATA / "nasa", export_path=PATH_EXPORT, normalize=False)
train_series = dataset.build_train_dataset()
test_series = dataset.build_test_dataset()
test_labels = dataset.build_test_labels()

print(
    f"SHAPES: TRAIN:{train_series[0].shape}\nTEST_SERIES: {test_series[0].shape}\nLABELS: {test_labels[0].shape}"
)

Carregando StrokeDataset...
INFO:root:Loaded 105 anomaly sequences across 81 channels.
INFO:root:Empty DataFrame
Columns: [value]
Index: []
INFO:root:DF TEST: feature_id series_id  time_idx    feat_0  target
0                A-1         0  1.000000       0
1                A-1         1  1.000000       0
2                A-1         2  1.000000       0
3                A-1         3  1.000000       0
4                A-1         4  1.000000       0
...              ...       ...       ...     ...
510220           T-9      1091  0.499149       0
510221           T-9      1092  0.501221       0
510222           T-9      1093  0.501221       0
510223           T-9      1094  0.501221       0
510224           T-9      1095 -0.954212       0

[510225 rows x 4 columns]
INFO:root:COLUNAS: Index(['series_id', 'time_idx', 'feat_0', 'target'], dtype='str', name='feature_id')
SHAPES: TRAIN:(8640, 1, 1)
TEST_SERIES: (8640, 1, 1)
LABELS: (8640,)

✓ Dataset carregado com sucesso!
TAMANHO DO DATASET:

## Exploratory Data Analysis

In [3]:
# ============================================================
# EXPLORAÇÃO E VISUALIZAÇÃO
# ============================================================

import numpy as np
import plotly.express as px
from statsmodels.tsa.stattools import acf, pacf
from darts.datasets.datasets import TimeSeries


# Função para calcular estatísticas de cada série
def calculate_series_stats(series: TimeSeries):
    """Calcula estatísticas para uma série temporal Darts."""
    # Converte para numpy array
    values_nan = series.values().flatten()

    values = np.array([elem for elem in values_nan if elem != np.nan])

    # Estatísticas básicas
    mean = np.mean(values)
    std = np.std(values)
    q1 = np.percentile(values, 25)
    q2 = np.percentile(values, 50)  # mediana
    q3 = np.percentile(values, 75)

    # Porcentagem de NaN
    pct_nan = (np.isnan(values).sum() / len(values)) * 100

    # Elementos acima de 3 desvios padrão
    above_3std = np.sum(np.abs(values - mean) > 3 * std)

    # ACF e PACF (até 20 lags ou menos se a série for menor)
    max_lags = min(20, len(values) // 2)
    if max_lags > 1:
        acf_values = acf(values, nlags=max_lags, fft=True)
        pacf_values = pacf(values, nlags=max_lags)
    else:
        acf_values = np.array([np.nan])
        pacf_values = np.array([np.nan])

    return {
        "mean": mean,
        "std": std,
        "q1": q1,
        "q2": q2,
        "q3": q3,
        "pct_nan": pct_nan,
        "above_3std": above_3std,
        "acf": acf_values,
        "pacf": pacf_values,
        "length": len(values),
    }


# Itera sobre todas as séries de treino
print("Calculando estatísticas para cada série...")
stats_list = []

for idx, series in enumerate(dataset.train_series):
    stats = calculate_series_stats(series)
    stats["series_id"] = idx
    stats_list.append(stats)

    if (idx + 1) % max(1, int(len(dataset.train_series) / 10)) == 0:
        print(f"Progresso: {((idx + 1) / len(dataset.train_series)) * 100:.1f}%")

print(f"Total de séries processadas: {len(stats_list)}")

# Cria DataFrame principal
df_stats = pd.DataFrame(
    [{k: v for k, v in s.items() if k not in ["acf", "pacf"]} for s in stats_list]
)

# Separa ACF e PACF em colunas
df_acf = pd.DataFrame(
    [s["acf"] for s in stats_list],
    columns=[f"acf_lag{i}" for i in range(len(stats_list[0]["acf"]))],
)
df_pacf = pd.DataFrame(
    [s["pacf"] for s in stats_list],
    columns=[f"pacf_lag{i}" for i in range(len(stats_list[0]["pacf"]))],
)

# Concatena tudo
df_stats = pd.concat([df_stats, df_acf, df_pacf], axis=1)

print(f"DataFrame de estatísticas criado: {df_stats.shape}")
print(df_stats.head())


# ============================================================
# VISUALIZAÇÕES
# ============================================================

# Heatmap: média +- desvio padrão das séries (bin=100)
# Criar bins de 100 em 100 séries
n_series = len(df_stats)
n_bins = 100
bin_size = n_series // n_bins

# Calcular média e std por bin
heatmap_data = []
for i in range(n_bins):
    start_idx = i * bin_size
    end_idx = min((i + 1) * bin_size, n_series)
    bin_series = df_stats.iloc[start_idx:end_idx]

    mean_val = bin_series["mean"].mean()
    std_val = bin_series["std"].mean()

    heatmap_data.append(
        {
            # "bin": i,
            "mean": mean_val,
            "std": std_val,
        }
    )

df_heatmap = pd.DataFrame(heatmap_data)

# Heatmap usando plotly
fig_heatmap = px.imshow(
    df_heatmap.T.values,
    labels=dict(x="Bin de séries", y="Valor", color="Valor"),
    x=[f"Bin {i}" for i in range(n_bins)],
    y=df_heatmap.columns.tolist(),
    title=f"Heatmap: Média ± Desvio Padrão das Séries (bins de {bin_size} séries)",
    color_continuous_scale="Viridis",
    aspect="auto",
)
fig_heatmap.update_layout(width=1200, height=400)
fig_heatmap.show()

# Scatter plot: % de NaN por série
fig_scatter = px.scatter(
    df_stats,
    x="series_id",
    y="pct_nan",
    title="% de NaN por Série",
    labels={"series_id": "ID da Série", "pct_nan": "% NaN"},
    hover_data=["length"],
)
fig_scatter.update_traces(marker=dict(size=4, opacity=0.6))
fig_scatter.update_layout(width=1200, height=400)
fig_scatter.show()

# Estatísticas resumidas
print("\n=== RESUMO DAS ESTATÍSTICAS ===")
print(f"Média global: {df_stats['mean'].mean():.4f}")
print(f"Desvio padrão médio: {df_stats['std'].mean():.4f}")
print(f"Séries com > 0% NaN: {(df_stats['pct_nan'] > 0).sum()}")
print(f"Máximo % NaN: {df_stats['pct_nan'].max():.1f}%")
print(f"Média de elementos acima de 3 std: {df_stats['above_3std'].mean():.2f}")

Calculando estatísticas para cada série...


AttributeError: 'NasaDataset' object has no attribute 'train_series'

# Treinamento dos modelos

## Modelos de baseline

In [ ]:
from src.models.outlier import OutlierDetector
from src.models.baselines import Hampel, KMeans, IsolationForest, LocalOutlierFactor

models: list[OutlierDetector] = [Hampel(), KMeans(), IsolationForest(), LocalOutlierFactor()]
print("\n✓ Dataset carregado com sucesso!")
print(f"TAMANHO DO DATASET: {dataset.tam_dataset}")
# =====================================================
# TRAIN
# =====================================================
metricas_models = []
for model in models:
    dict_met = model.apply(train_series, test_series, test_labels)
    metricas_models.append(dict_met)
    gc.collect()
res = pd.DataFrame(metricas_models)
print(res)

## Modelos Rede Neural

In [ ]:
from src.models.nn.base_model import OutlierModelWrapper
from src.models.nn import TCN, VAE

WINDOW_SIZE = 20

wrappers: list[OutlierModelWrapper] = [
    OutlierModelWrapper(
        input_dim=WINDOW_SIZE,
        pl_model=TCN(input_dim=WINDOW_SIZE),
        window_size=WINDOW_SIZE,
        lr=1e-3,
        batch_size=32,
        max_epochs=5,
    ),
    OutlierModelWrapper(
        input_dim=WINDOW_SIZE,
        pl_model=VAE(input_dim=WINDOW_SIZE),
        window_size=WINDOW_SIZE,
        lr=1e-3,
        batch_size=32,
        max_epochs=5,
    ),
]

print("\n✓ Dataset carregado com sucesso!")
print(f"TAMANHO DO DATASET: {dataset.tam_dataset}")
# =====================================================
# TRAIN
# =====================================================
metricas_models = []
for wrapper in wrappers:
    dict_met = wrapper.apply(train_series, test_series, test_labels)
    metricas_models.append(dict_met)
    gc.collect()
res = pd.DataFrame(metricas_models)
print(res)

## Teste dos modelos

## Otimização de hiperparâmetros

In [ ]:
import gc
import logging
import os
from pathlib import Path
from typing import Literal

import mlflow
import torch
from lightning import Trainer, seed_everything
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import MLFlowLogger
from mlflow.pytorch import autolog
from src.models.kan import KANSearchSpace, MyKan
from src.models.mlp import MLP, MLPSearchSpace


def get_best_hyperparameters_from_optuna(choice: str, mlf_track_uri: str) -> dict:
    """
    Reads the best run ID from the CSV file saved during Optuna training,
    fetches the run from MLflow, and returns the hyperparameters as a dict.

    Args:
        choice: Model architecture name (e.g., "MLP", "KAN")
        mlf_track_uri: MLflow tracking URI

    Returns:
        Dictionary of hyperparameters from the best Optuna run (with string keys)
    """
    return {}  # Placeholder - implement as needed


def zip_res_simplified(
    export_dir: Path,
    filename: str,
    dest_folder: Path | None = None,
    mlflow_db_path: Path | None = None,
    optuna_db_path: Path | None = None,
    mlruns_path: Path | None = None,
):
    """
    Simplified zip function that copies databases and mlruns to export_dir, then zips everything.

    Args:
        export_dir: Directory containing all files to zip (artifacts, CSVs, etc.)
        filename: Name of the output zip file
        dest_folder: Destination folder for the zip file
        mlflow_db_path: Path to mlflow.db to copy into export_dir
        optuna_db_path: Path to optuna.db to copy into export_dir
        mlruns_path: Path to mlruns folder to copy into export_dir
    """
    import shutil

    # Ensure export_dir exists
    export_dir.mkdir(parents=True, exist_ok=True)

    # Copy databases if they exist
    if mlflow_db_path and mlflow_db_path.exists():
        shutil.copy(mlflow_db_path, export_dir / mlflow_db_path.name)
        print(f"Copied {mlflow_db_path.name} to export directory")

    if optuna_db_path and optuna_db_path.exists():
        shutil.copy(optuna_db_path, export_dir / optuna_db_path.name)
        print(f"Copied {optuna_db_path.name} to export directory")

    # Copy mlruns folder if it exists
    if mlruns_path and mlruns_path.exists():
        export_mlruns = export_dir / mlruns_path.name
        if export_mlruns.exists():
            shutil.rmtree(export_mlruns)
        shutil.copytree(mlruns_path, export_mlruns)
        print(f"Copied {mlruns_path.name} folder to export directory")

    # Determine destination folder
    if dest_folder is None:
        dest_folder = Path.cwd()
    else:
        dest_folder.mkdir(parents=True, exist_ok=True)

    # Create zip file
    zip_path = dest_folder / filename.replace(".zip", "")
    shutil.make_archive(str(zip_path), "zip", export_dir)
    print(f"PATH ZIPFILE: {zip_path.with_suffix('.zip').resolve()}")


def supress_warnings():
    # Suppress specific MLflow warnings
    logging.getLogger("mlflow.utils.requirements_utils").setLevel(logging.ERROR)
    logging.getLogger("mlflow").setLevel(logging.ERROR)

    # Suppress PyTorch Lightning info messages
    logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.ERROR)


def model_choice(
    CHOICE,
    INPUT_DIMS,
    N_CLASSES,
    recall_factor: list[float] | None = None,
    optuna_hyperparams: dict[str, float | int] = {},
):
    """
    Seleciona e configura o modelo baseado na escolha.

    Args:
        CHOICE: Arquitetura do modelo ("MLP" ou "KAN")
        INPUT_DIMS: Dimensionalidade da entrada
        N_CLASSES: Número de classes
        recall_factor: Fator de recall para balanceamento (opcional)
        optuna_hyperparams: Hiperparâmetros do Optuna

    Returns:
        Tuple de (model, suggested_hparams, keys)
    """
    if recall_factor is None:
        recall_factor = [1.0, 1.0]

    if CHOICE == "MLP":
        search_space = MLPSearchSpace()

        keys = search_space.Keys
        hyperparams = {
            keys.BATCH_SIZE: 32,
            keys.HIDDEN_DIMS: 512,
            keys.LR: 2e-7,
            keys.WEIGHT_DECAY: 1e-5,
            keys.BETA0: 0.99,
            keys.BETA1: 0.999,
            keys.N_LAYERS: 80,
        }

        # Map optuna hyperparams (string keys) to Keys enum
        if optuna_hyperparams:
            mapped_hyperparams = {}
            for key_enum, default_value in hyperparams.items():
                # Get the string representation of the enum key
                key_str = key_enum.value
                if key_str in optuna_hyperparams:
                    mapped_hyperparams[key_enum] = optuna_hyperparams[key_str]
                else:
                    mapped_hyperparams[key_enum] = default_value
            hyperparams = mapped_hyperparams

        print(f"MLP HYPERPARAMS: {hyperparams}")
        suggested_hparams = search_space.suggest(hyperparams)
        model = MLP(
            INPUT_DIMS,
            N_CLASSES,
            recall_factor=recall_factor,
            hyperparameters=suggested_hparams,
        )
    elif CHOICE == "KAN":
        search_space = KANSearchSpace()
        keys = search_space.Keys
        hyperparams = {
            keys.BATCH_SIZE: 32,
            keys.HIDDEN_DIMS: 100,
            keys.LR: 2e-7,
            keys.WEIGHT_DECAY: 1e-5,
            keys.BETA0: 0.99,
            keys.BETA1: 0.999,
            keys.GRID: 184,
            keys.SPLINE_POL_ORDER: 4,
        }

        # Map optuna hyperparams (string keys) to Keys enum
        if optuna_hyperparams:
            mapped_hyperparams = {}
            for key_enum, default_value in hyperparams.items():
                # Get the string representation of the enum key
                key_str = key_enum.value
                if key_str in optuna_hyperparams:
                    mapped_hyperparams[key_enum] = optuna_hyperparams[key_str]
                else:
                    mapped_hyperparams[key_enum] = default_value
            hyperparams = mapped_hyperparams

        print(f"KAN HYPERPARAMS: {hyperparams}")
        suggested_hparams = search_space.suggest(hyperparams)
        model = MyKan(
            INPUT_DIMS,
            N_CLASSES,
            recall_factor=recall_factor,
            hyperparameters=suggested_hparams,
        )
    else:
        raise ValueError("ESCOLHA DE MODELO ERRADA!")
    return model, suggested_hparams, keys


## ======================== FUNÇÃO PRINCIPAL ========================


def main(CHOICE: str):
    """
    Função principal de treinamento.

    Args:
        CHOICE: Arquitetura do modelo ("MLP" ou "KAN")
    """
    ###------SEEDS---------###
    RAND_SEED = 42
    seed_everything(RAND_SEED)
    supress_warnings()

    AMBIENTE = os.environ["AMBIENTE"]
    GPU = True if AMBIENTE in ["KAGGLE", "COLAB"] else False

    ## ---------- VARIÁVEIS DE TREINAMENTO -----------
    cpus = os.cpu_count()
    WORKERS = cpus if cpus is not None else 1
    NUM_DEVICES = 1 if GPU else 1
    NUM_NODES = 1
    BATCH_SIZE = 64
    EPOCHS = 1
    PATIENCE = int(EPOCHS * 0.6)
    ARTIFACT_PATH = EXPORT_DIR / "artifacts"
    os.makedirs(ARTIFACT_PATH, exist_ok=True)

    #### -------- VARIÁVEIS DE LOGGING ------------
    EXP_NAME = "PROD_TRAINING"
    RUN_NAME: str | None = f"PROD_{CHOICE}"
    MLF_TRACK_URI = f"sqlite:///{PATH_CODE}/mlflow.db"

    mlflow.set_tracking_uri(MLF_TRACK_URI)
    mlflow.set_experiment(EXP_NAME)
    autolog(log_models=True, checkpoint=True, exclusive=False)

    hyperparams = get_best_hyperparameters_from_optuna(CHOICE, MLF_TRACK_URI)

    ## ---------- VARIÁVEIS DO MODELO -----------
    N_CLASSES = 2

    # Preparar datamodule
    datamodule = StrokeDataModule(BATCH_SIZE, WORKERS)
    datamodule.prepare_data()
    datamodule.setup("fit")

    INPUT_DIMS = datamodule.input_dims or -1
    assert INPUT_DIMS > 0 and INPUT_DIMS is not None

    # Criar modelo
    model, _, keys = model_choice(
        CHOICE,
        INPUT_DIMS,
        N_CLASSES,
        recall_factor=None,  # Sem balanceamento específico por enquanto
        optuna_hyperparams=hyperparams,
    )

    _ = model(model.example_input_array)

    # Loop principal de treinamento
    with mlflow.start_run(run_name=RUN_NAME) as run:
        active_run_id = run.info.run_id

        mlflow_logger = MLFlowLogger(
            experiment_name=EXP_NAME,
            tracking_uri=MLF_TRACK_URI,
            log_model=True,
            run_id=active_run_id,
        )

        early_stopping = EarlyStopping(monitor="val_loss", patience=PATIENCE, mode="min")

        trainer = Trainer(
            max_epochs=EPOCHS,
            devices=NUM_DEVICES,
            accelerator="gpu" if GPU else "cpu",
            num_nodes=NUM_NODES,
            logger=mlflow_logger,
            enable_checkpointing=False,
            log_every_n_steps=1,
            callbacks=[early_stopping],
        )
        trainer.fit(model, datamodule=datamodule)
        mlflow.log_params(dict(model.hparams))

        # Test the model
        test_loader = datamodule.test_dataloader()
        trainer.test(model, dataloaders=test_loader)

        torch.cuda.empty_cache()

    return


if __name__ == "__main__":
    try:
        ARQ_TYPE = Literal["MLP", "KAN", "SVM", "XGBOOST"]  ## MODEL ARCHITECTURE
        models: list[ARQ_TYPE] = ["MLP", "KAN"]
        for choice in models:
            # trains model based on architecture
            if choice == "MLP":
                continue
            main(choice)

        NAME_RESZIP = "resultado_kaggle_stroke_normal"
        MLRUNS_FOLDER = PATH_CODE / "mlruns"
        MLF_TRACK_URI = f"sqlite:///{PATH_CODE}/mlflow.db"
        OPTUNA_DB_PATH = PATH_CODE / "optuna.db"
        ZIP_ROOT = PATH_DATASET / ".." if os.environ["AMBIENTE"] == "KAGGLE" else PATH_DATASET

        zip_res_simplified(
            export_dir=EXPORT_DIR,
            filename=NAME_RESZIP,
            dest_folder=ZIP_ROOT,
            mlflow_db_path=Path(MLF_TRACK_URI.replace("sqlite:///", "")),
            optuna_db_path=OPTUNA_DB_PATH,
            mlruns_path=MLRUNS_FOLDER,
        )
        print("\n", "=" * 60)
        print(f"RESULTADOS ZIPADOS {Path(ZIP_ROOT, NAME_RESZIP).resolve()}")
        print("=" * 60, "\n")

    except Exception as e:
        raise e
    gc.collect()

    if os.environ["AMBIENTE"] == "LOCAL":
        from view.dashboard import see_model

        see_model(PATH_DATASET / "mlflow.db", PATH_DATASET / ".." / "mlruns")

## Teste da otimização

In [ ]:
# ## OUTPUT: MLFlow's Dashboard (Only works outside of Kaggle)
# ### Download the training results from Kaggle and paste them into a cloned folder of the repository
# # ============================================================
# # VISUALIZAÇÕES
# # ============================================================

# # Heatmap: média +- desvio padrão das séries (bin=100)
# # Criar bins de 100 em 100 séries
# n_series = len(df_stats)
# n_bins = 100
# bin_size = n_series // n_bins

# # Calcular média e std por bin
# heatmap_data = []
# for i in range(n_bins):
#     start_idx = i * bin_size
#     end_idx = min((i + 1) * bin_size, n_series)
#     bin_series = df_stats.iloc[start_idx:end_idx]

#     mean_val = bin_series["mean"].mean()
#     std_val = bin_series["std"].mean()

#     heatmap_data.append(
#         {
#             "bin": i,
#             "mean": mean_val,
#             "std": std_val,
#             "mean_minus_std": mean_val - std_val,
#             "mean_plus_std": mean_val + std_val,
#         }
#     )

# df_heatmap = pd.DataFrame(heatmap_data)

# # Heatmap usando plotly
# fig_heatmap = px.imshow(
#     df_heatmap[["mean_minus_std", "mean", "mean_plus_std"]].T.values,
#     labels=dict(x="Bin de séries", y="Valor", color="Valor"),
#     x=[f"Bin {i}" for i in range(n_bins)],
#     y=["mean - std", "mean", "mean + std"],
#     title=f"Heatmap: Média ± Desvio Padrão das Séries (bins de {bin_size} séries)",
#     color_continuous_scale="Viridis",
# )
# fig_heatmap.update_layout(width=1200, height=400)
# fig_heatmap.show()

# # Scatter plot: % de NaN por série
# fig_scatter = px.scatter(
#     df_stats,
#     x="series_id",
#     y="pct_nan",
#     title="% de NaN por Série",
#     labels={"series_id": "ID da Série", "pct_nan": "% NaN"},
#     hover_data=["length"],
# )
# fig_scatter.update_traces(marker=dict(size=4, opacity=0.6))
# fig_scatter.update_layout(width=1200, height=400)
# fig_scatter.show()

# # Estatísticas resumidas
# print("\n=== RESUMO DAS ESTATÍSTICAS ===")
# print(f"Média global: {df_stats['mean'].mean():.4f}")
# print(f"Desvio padrão médio: {df_stats['std'].mean():.4f}")
# print(f"Séries com > 0% NaN: {(df_stats['pct_nan'] > 0).sum()}")
# print(f"Máximo % NaN: {df_stats['pct_nan'].max():.1f}%")
# print(f"Média de elementos acima de 3 std: {df_stats['above_3std'].mean():.2f}")